#### 크롤링 예제
1. requests lib 이용해서 'moons-86.iptime.org:8080' 요청 
2. 응답 받은 데이터를 BeautifulSoup을 이용하여 데이터를 파싱
3. id가 product_1001인 태그를 찾아서 h2, p의 모든 콘텐츠 데이터를 추출한다.
4. 모든 상품의 정보를 추출
    - 데이터프레임으로 생성
    - csv 파일로 저장


In [42]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
from datetime import datetime


res = requests.get("http://moons-86.iptime.org:8080")
res



<Response [200]>

In [43]:
# 1001
soup = bs(res.text, 'html.parser')
product = soup.find("div", id="product-1003")
print(product)

<div class="product-item" id="product-1003">
<h2 class="product-title">고급 가죽 지갑</h2>
<p class="product-category">카테고리: 패션 잡화</p>
<p class="product-price">가격: 75,000원</p>
<p class="product-rating">평점: ★★★★☆ (4.0/5)</p>
<a class="product-link" href="/products/1003">상세 보기</a>
</div>


In [55]:
# 모든 상품 정보
data_list = []
products_select = soup.select("div[id^='product-']")

for selector in products_select:
    data_list.append({
        "id" : selector.get("id"),
        "상품명" : selector.find(class_="product-title").get_text(strip=True),
        "카테고리" : selector.find(class_="product-category").get_text(strip=True).replace("카테고리: ", ""),
        "가격" : selector.find(class_="product-price").get_text(strip=True).replace("가격: ", ""),
        "평점" : selector.find(class_="product-rating").get_text(strip=True).replace("평점: ", ""),
        "링크" : "http://moons-86.iptime.org:8080" + selector.find(class_="product-link")["href"]
    })

df = pd.DataFrame(data_list)

now = datetime.now()
now_str = now.strftime('%y-%m-%d')

df.to_csv("./products_"+now_str+".csv", index=False, encoding="cp949")